In [4]:
import os
from typing import Annotated, List, TypedDict, Optional, Union
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\ASUS\OneDrive\Desktop\DeepForge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# The specialized framework for deep reasoning agents
from deepagents import create_deep_agent

In [6]:
load_dotenv()

True

In [7]:
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    paper_path: Optional[str]
    scientific_logic: Optional[str]
    implementation_code: Optional[str]

print("Environment and State initialized.")

Environment and State initialized.


In [8]:
llm= ChatGoogleGenerativeAI(
    model= "gemini-2.0-flash",
    temperature= 0
)

In [9]:
research_agent= create_deep_agent(
    model= llm,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your job is to read research papers, extract the mathematical core, "
        "and explain the architecture in plain pseudocode. "
        "Use your planning tools to break down the paper analysis step-by-step."
    )
)
print("Deep Research Agent created.")

Deep Research Agent created.


In [10]:
import arxiv
import fitz  
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

In [11]:
# 1. arXiv Search and Download Tool
@tool
def download_research_paper(query: str)-> str:
    """
    Searches arXiv for a paper, downloads the PDF to the '../data/' folder, 
    and returns the local path to the file.
    """
    download_path= "../data/"
    os.makedirs(download_path, exist_ok= True)

    search= arxiv.Search(query= query, max_results= 1, sort_by= arxiv.SortCriterion.Relevance)
    paper= next(search.results())

    filename = f"{paper.title.replace(' ', '_').replace(':', '')}.pdf"
    full_path = paper.download_pdf(dirpath=download_path, filename=filename)
    
    return f"Paper downloaded to: {full_path}"

In [12]:
# 2. PDF Reading Tool
@tool
def read_pdf_content(file_path: str) -> str:
    """
    Opens a PDF file and extracts the text content. 
    Useful for reading downloaded research papers.
    """
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [13]:
# 3. Web Search Tool (Tavily)
web_search = TavilySearchResults(k=3)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_1092\3970997022.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(k=3)


In [14]:
# Combine all tools
tools = [download_research_paper, read_pdf_content, web_search]
print("ArXiv, PDF, and Tavily tools defined.")

ArXiv, PDF, and Tavily tools defined.


In [15]:
# 1. Re-initialize the Researcher Agent with tools
# We give it our 'tools' list so it can search, download, and read.
researcher_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your mission is to find research papers and extract their implementation core. "
        "Step 1: Use 'download_research_paper' to get the PDF. "
        "Step 2: Use 'read_pdf_content' to understand the math and logic. "
        "Step 3: Output a structured explanation of the algorithm, variables, and pseudocode."
    )
)
print("Research Agent re-Linked with tools.")

Research Agent re-Linked with tools.


In [16]:
# trigger a "Deep" research session
inputs = {
    "messages": [
        ("user", "Research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.")
    ]
}

config = {"configurable": {"thread_id": "research-task-1"}}
print("Starting Deep Research. Watch the agent plan and execute...")

for chunk in research_agent.stream(inputs, config):
    print(chunk)

Starting Deep Research. Watch the agent plan and execute...
{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.', additional_kwargs={}, response_metadata={}, id='d0187856-03b9-46f5-b6c3-6216d06b558f')])}}
{'model': {'messages': [AIMessage(content='Okay, I will research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.', additional_kwargs={'function_call': {'name': 'write_todos', 'arguments': '{"todos": [{"content": "Research the QLoRA paper to understand the 4-bit NormalFloat quantization method.", "status": "in_progress"}, {"status": "pending", "content": "Explain the 4-bit NormalFloat quantization method in detail."}]}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cfa23-bbfb-72a1-836b-8b52cab27316-0', tool_calls=[{'name': 'write_to

In [17]:
# This agent takes the research findings and plans the Code structure.
designer_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt= (
        "You are the Architecture Designer. "
        "Your goal is to take research findings and create a high-level Design Spec. "
        "1. Define the Python classes and methods needed. "
        "2. Specify the exact mathematical formulas for each method. "
        "3. Decide on the libraries to use (e.g., torch, numpy). "
        "Output ONLY the design specification, not the actual code yet."
    )
)
print("Architecture Designer Agent created.")

Architecture Designer Agent created.


In [18]:
workflow = StateGraph(ResearchState)

# 1. Add Nodes
# IMPORTANT: We use .invoke()["messages"] to return a dict that matches our State
workflow.add_node("researcher", lambda state: {"messages": researcher_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("designer", lambda state: {"messages": designer_agent.invoke({"messages": state["messages"]})["messages"]})

# 2. Define the Flow (Initial)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "designer")
workflow.add_edge("designer", END)

# 3. Compile
app = workflow.compile()
print("Node logic fixed for Researcher & Designer.")

Node logic fixed for Researcher & Designer.


In [19]:
# The Coder Agent
# Its only job is to turn the Design Spec into high-quality, executable Python code.
coder_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt=(
        "You are the Lead Programmer for DeepForge. "
        "Your goal is to implement the Architecture Designer's specification. "
        "1. Write complete, well-documented Python code. "
        "2. Include a 'main' block or test case to verify the implementation. "
        "3. Ensure you follow research paper naming conventions for variables. "
        "Output ONLY the Python code inside a markdown block."
    )
)
print("Coder Agent created.")

Coder Agent created.


In [20]:
# 1. Add the Coder node to the existing workflow
workflow.add_node("coder", lambda state: {"messages": coder_agent.invoke({"messages": state["messages"]})["messages"]})

# 2. Re-route the Designer to the Coder instead of END
# We use a new edge to extend the chain
workflow.add_edge("designer", "coder")
workflow.add_edge("coder", END)

# 3. Re-compile the Full Pipeline
deep_agent_pipeline = workflow.compile()
print("✅ Pipeline evolved! Coder added successfully.")

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


✅ Pipeline evolved! Coder added successfully.


In [21]:
final_inputs= {
    "messages": [
        ("user", "Research the 'LoRA: Low-Rank Adaptation' paper and implement a simple LoRA linear layer in PyTorch.")
    ]
}
print("Executing DEEP PIPELINE: Reasearching-> Designing-> Coding...")

# Run the pipeline
for event in deep_agent_pipeline.stream(final_inputs, config):
    # This will show you exactly which agent is working
    for node_name, output in event.items():
        print(f"\n--- [AGENT: {node_name.upper()}] ---")
        # We print the last message content from that agent
        print(output["messages"][-1].content)

Executing DEEP PIPELINE: Reasearching-> Designing-> Coding...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_1092\2121793795.py:12: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  paper= next(search.results())



--- [AGENT: RESEARCHER] ---
Here's a breakdown of the LoRA implementation:

**Algorithm:**

LoRA approximates weight updates by decomposing them into low-rank matrices. Instead of directly training the weights of a large neural network layer, LoRA adds a smaller, low-rank matrix to the original weights. This reduces the number of trainable parameters and makes training more efficient.

**Variables:**

*   `in_features`: The number of input features to the linear layer.
*   `out_features`: The number of output features from the linear layer.
*   `r`: The rank of the low-rank matrices. This determines the number of trainable parameters.
*   `lora_A`: A matrix of shape (in\_features, r).
*   `lora_B`: A matrix of shape (r, out\_features).
*   `scaling`: A scaling factor applied to the low-rank matrices.

**Pseudocode:**

```
class LoRALinear:
  def __init__(in_features, out_features, r):
    self.A = random_matrix(in_features, r)
    self.B = zero_matrix(r, out_features)
    self.scaling

In [24]:
from langchain_experimental.utilities import PythonREPL
from langchain_core.tools import tool 

# Initialize the Python Runtime
repl = PythonREPL()

@tool
def execute_python_code(code: str) -> str:
    """
    Executes Python code and returns the console output. 
    Use this to verify if the generated implementation actually runs.
    """
    try:
        # We strip backticks if the agent included them
        clean_code = code.replace("```python", "").replace("```", "")
        # FIX: Changed 'replace' to 'repl'
        result = repl.run(clean_code)
        return f"Execution Success:\n{result}"
    except Exception as e:
        return f"Execution Failed:\n{str(e)}"

# Add the new tool to our collection
tools.append(execute_python_code)

print("Execution Tool fixed and added.")

Execution Tool fixed and added.


In [26]:
# Master Pipeline Setup - Run this to rebuild the whole graph
workflow = StateGraph(ResearchState)

# 1. Define all nodes
workflow.add_node("researcher", lambda state: {"messages": researcher_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("designer", lambda state: {"messages": designer_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("coder", lambda state: {"messages": coder_agent.invoke({"messages": state["messages"]})["messages"]})

# Executor node logic
def executor_node(state: ResearchState):
    last_message = state["messages"][-1].content
    if "```python" in last_message:
        code = last_message.split("```python")[1].split("```")[0]
        result = execute_python_code.invoke(code)
        return {"messages": [HumanMessage(content=f"EXECUTOR RESULTS:\n{result}")]}
    return {"messages": [HumanMessage(content="ERROR: No code block found to execute.")]}

workflow.add_node("executor", executor_node)

# 2. Define the Flow
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "designer")
workflow.add_edge("designer", "coder")
workflow.add_edge("coder", "executor")

# Conditional logic for self-healing
def should_continue(state: ResearchState):
    last_message = state["messages"][-1].content
    if "Execution Success" in last_message:
        return END
    return "coder"

workflow.add_conditional_edges("executor", should_continue)

# 3. Compile
deep_agent_pipeline = workflow.compile()
print("Master Pipeline compiled successfully!")

Master Pipeline compiled successfully!


In [27]:
# The Reviewer Agent
reviewer_agent= create_deep_agent(
    model= llm,
    system_prompt= (
        "You are the Peer Reviewer. "
        "Your job is to audit the generated code against the original research logic. "
        "1. Check if the mathematical formulas match the paper. "
        "2. Look for missing initialization steps (e.g., zero-init for weights). "
        "3. Verify if variable names match the paper's notation. "
        "Output a 'Review Report'. If the code is perfect, say 'LGTM' (Looks Good To Me). "
        "If there are errors, explain them clearly so the Coder can fix them."
    )
)
print("Reviewer Agent created.")

Reviewer Agent created.
